In [16]:
import os
import zipfile
import shutil
from urllib import request

In [2]:
%pwd

'c:\\Users\\Its KaMrAN\\Documents\\Linux Docker\\Wine Prediction ML\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Its KaMrAN\\Documents\\Linux Docker\\Wine Prediction ML'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [9]:
from Wine_Prediction_ML.constants import *
from Wine_Prediction_ML.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [21]:
import os
import zipfile
import urllib.request as request
from pathlib import Path
from src.Wine_Prediction_ML import logger
from src.Wine_Prediction_ML.utils.common import get_size

In [24]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        # Check if it's a zip file or CSV file
        if self.config.local_data_file.endswith('.zip'):
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
        else:
            # For CSV files, just copy to the destination if not the same file
            import shutil
            destination = os.path.join(unzip_path, os.path.basename(self.config.local_data_file))
            # Check if source and destination are not the same file
            if os.path.abspath(self.config.local_data_file) != os.path.abspath(destination):
                shutil.copy(self.config.local_data_file, destination)
                logger.info(f"Copied {self.config.local_data_file} to {destination}")
            else:
                logger.info(f"File already in destination: {destination}")

In [25]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-14 08:04:19,699: INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-03-14 08:04:19,703: INFO:common:yaml file: params.yaml loaded successfully]
[2026-03-14 08:04:19,709: INFO:common:yaml file: schema.yaml loaded successfully]
[2026-03-14 08:04:19,712: INFO:common:created directory at: artifacts]
[2026-03-14 08:04:19,715: INFO:common:created directory at: artifacts/data_ingestion]
[2026-03-14 08:04:19,717: INFO:4159884592:File already exists of size: ~ 76 KB]
[2026-03-14 08:04:19,726: INFO:4159884592:File already in destination: artifacts/data_ingestion\WineQT.csv]
